# Phase 7 - Stage 2 B fine-tune (warm-start + low LR)

**Goal**: Fine-tune Stage 2 from the Phase 6 warm-start with the new dual-path
mu_total cache, using config B from the smoke (RMSE delta 0.10 at ep1, plateau
on 3 ep) to recover skill while keeping causality.

**Setup**:
  - Stage 1 dual-path : FROZEN (epoch_best_dualpath.pth)
  - Stage 2 init      : warm-start from oracle_9node epoch_last.pth
  - Optimizer         : AdamW lr=5e-5, weight_decay=1e-4, no warmup
  - Loss              : train_epoch_stage2_cached (bf16 on A100, EDM Karras weighting)
  - EMA               : decay 0.9999 (standard Karras EDM2)
  - Epochs            : 50 (~2-3h on A100 with local SSD cache)

**Eval**: BS30 protocol on test split via phase6_dualpath_final_validation.ipynb
pointed at the new checkpoint (separate notebook, runs in ~30-45 min).

**Persist** :
  - epoch_last.pth   : updated every epoch (state_dict + EMA + optimizer + history)
  - epoch_best.pth   : updated when train_loss reaches new minimum
  - training_history.json : flat list of per-epoch records


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab - Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Constants (config B fine-tune) ===
import json
import numpy as np
import torch
from pathlib import Path
from omegaconf import OmegaConf

# ----- Drive paths -----
DRIVE_ROOT       = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N        = DRIVE_ROOT / 'oracle_9node' / 'seed_42'

# Stage 1 dual-path (frozen).
CKPT_DUALPATH    = ORACLE_9N / 'epoch_best_dualpath.pth'

# Warm-start source : Phase 6 Stage 2 (after the smoke proved this is the best init).
CKPT_STAGE2_WARMSTART = ORACLE_9N / 'epoch_last.pth'

# Cached Stage 1 outputs for Stage 2 training (mu_total via dual_path).
STAGE1_CACHE_PATH = ORACLE_9N / 'stage1_cache_phase7_dualpath.pt'

# Output dir for B fine-tune artefacts.
OUT_DIR = ORACLE_9N / 'phase7_stage2_B_finetune'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_LAST      = OUT_DIR / 'epoch_last.pth'
CKPT_BEST      = OUT_DIR / 'epoch_best.pth'
HISTORY_JSON   = OUT_DIR / 'training_history.json'

# Phase 6 dualpath-recalibrated sigma_data.
SIGMA_DATA_NEW = 0.193

# ----- Config B hyperparams (from smoke ranking : RMSE 0.10 dominant) -----
WARM_START                = True
LR                        = 5e-5
WEIGHT_DECAY              = 1e-4
BETA1, BETA2              = 0.9, 0.999
WARMUP_STEPS              = 0           # no warmup for warm-start
GRADIENT_CLIP             = 1.0
GRADIENT_CHECKPOINTING    = True
EMA_DECAY                 = 0.9999      # standard Karras EDM2
USE_AMP                   = True        # bf16 auto on A100 via train_epoch_stage2_cached

# Fine-tune budget (50 ep ~2-3h on A100 with cache).
EPOCHS_TARGET = 50

# Resume by default if epoch_last.pth in OUT_DIR.
RESUME = True

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE                = {DEVICE}')
print(f'[Cell 2] CKPT_DUALPATH         = {CKPT_DUALPATH}  exists={CKPT_DUALPATH.exists()}')
print(f'[Cell 2] CKPT_STAGE2_WARMSTART = {CKPT_STAGE2_WARMSTART}  exists={CKPT_STAGE2_WARMSTART.exists()}')
print(f'[Cell 2] STAGE1_CACHE_PATH     = {STAGE1_CACHE_PATH}  exists={STAGE1_CACHE_PATH.exists()}')
print(f'[Cell 2] OUT_DIR               = {OUT_DIR}')
print(f'[Cell 2] CKPT_LAST exists      = {CKPT_LAST.exists()}  (RESUME={RESUME})')
print(f'[Cell 2] LR={LR}  EMA_DECAY={EMA_DECAY}  GRAD_CLIP={GRADIENT_CLIP}  EPOCHS={EPOCHS_TARGET}')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config : merge base + corrdiff_normal (== non-causal stage 2 hyperparams) ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Stage 1 dataloader stays at BS=1 (we iterate samples once to fill the cache).
# Stage 2 will use a SEPARATE cached dataloader at BS=64 (see Cell 5).
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Load Stage 1 dual-path (encoder + RCN + head + dual_path) FROZEN ===
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder


def _parse_encoder_metapaths_from_ckpt(enc_sd):
    seen, order = {}, []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]; src = parts[1]; rel = parts[2]; tgt = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt); order.append(name)
    return [(n,) + seen[n] for n in order]


def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd


def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] loaded from "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] FAILED with "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] no valid key found in {keys}')
    return False


def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out


print(f'[Cell 4] Loading Stage 1 dual-path : {CKPT_DUALPATH}')
ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] ckpt keys[:12] = {sorted(ck_s1.keys())[:12]}')

enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
print(f'  metapaths detected : {[t[0] for t in parsed]}')
cfgs = [
    IntelligibleVariableConfig(name=n, meta_path=(s, r, t), pool='mean')
    for n, s, r, t in parsed
]
encoder = IntelligibleVariableEncoder(
    configs=cfgs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
n_vars = len(cfgs)
num_vars = n_vars

_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=n_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40
dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

_safe_load(encoder,         ck_s1, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_s1, ['dual_path_state_dict'],                          'dual_path')

# FREEZE all Stage 1 modules.
_stage1_n_total = 0
_stage1_n_trainable = 0
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        p.requires_grad_(False)
        _stage1_n_total += p.numel()
    m.eval()
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        if p.requires_grad:
            _stage1_n_trainable += p.numel()
assert _stage1_n_trainable == 0, (
    f'Stage 1 freeze failed : {_stage1_n_trainable} trainable params remain'
)
print(f'  [verify] Stage 1 frozen : {_stage1_n_total:,} params, '
      f'{_stage1_n_trainable} trainable (must be 0).')

_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'  A_dag shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')

print(f'[Cell 4] Stage 1 dual-path FROZEN. num_vars = {num_vars}')


# Helper used by precompute loop to compute mu_total via dual_path.
@torch.no_grad()
def predict_mu_total(_batch):
    '''Stage 1 forward (frozen): returns (mu_total, baseline_log, hr_residual).

    mu_total = mu_A + gate * mu_B(LR)   [B, 1, H, W]
    baseline_log = baseline at last step [B, 1, H, W]
    hr_residual  = HR_log1p - baseline   [B, 1, H, W]  (= sample['residual'][-1])
    '''
    lr_data = _batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(_batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A    = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3:
        mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(_batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, _mu_B, _gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    bl = _batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    hr_res = _batch['residual'][-1].to(DEVICE)
    if hr_res.dim() == 3:
        hr_res = hr_res.unsqueeze(0)
    return mu_total, bl, hr_res


In [ ]:
# === Cell 5 : Precompute mu_total cache + build cached dataloaders ===
# This cell runs the FROZEN Stage 1 dual-path once over the train_dataset,
# stores (mu_HR=mu_total, baseline_log, delta_target=residual - mu_total,
# valid_mask) on Drive, then exposes a map-style _CachedDataset + DataLoader
# at batch_size=64 -- reused by all 3 smoke configs.
import time

# Copy cache from Drive to local SSD (avoid mmap/page-fault stalls on Drive)
import shutil as _shutil_phase7
LOCAL_CACHE_PATH = Path('/content/stage1_cache_phase7_dualpath.pt')
if STAGE1_CACHE_PATH.exists() and (not LOCAL_CACHE_PATH.exists() or LOCAL_CACHE_PATH.stat().st_size != STAGE1_CACHE_PATH.stat().st_size):
    print(f'[Cell 5] Copying cache Drive -> local SSD (avoid mmap stall)...')
    import time as _t
    _t0 = _t.time()
    _shutil_phase7.copy(str(STAGE1_CACHE_PATH), str(LOCAL_CACHE_PATH))
    print(f'  Done in {_t.time()-_t0:.1f}s ({LOCAL_CACHE_PATH.stat().st_size/1e9:.2f} GB)')

if LOCAL_CACHE_PATH.exists():
    print(f'[Cell 5] Loading cache from local SSD {LOCAL_CACHE_PATH}')
    cache = torch.load(LOCAL_CACHE_PATH, map_location='cpu', weights_only=False)
    # Force materialize tensors into dense RAM (break any mmap/lazy backing)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
    _gb = sum(v.element_size()*v.nelement() for v in cache.values())/1e9
    print(f'[Cell 5] Cache rehydrated to dense RAM ({_gb:.2f} GB)')
elif STAGE1_CACHE_PATH.exists():
    print(f'[Cell 5] Loading existing cache {STAGE1_CACHE_PATH}')
    cache = torch.load(STAGE1_CACHE_PATH, map_location='cpu', weights_only=False)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
else:
    print('[Cell 5] Precomputing mu_total cache (one-time, ~30-45 min on A100)...')
    mu_HR_list, baseline_log_list, delta_target_list, valid_mask_list = [], [], [], []
    _t0 = time.time()

    _n_iter = len(train_dataset) if hasattr(train_dataset, '__len__') else None
    for i, sample in enumerate(train_dataset):
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        with torch.no_grad():
            mu_total, bl, hr_res = predict_mu_total(batch)
        # delta_target = (HR - baseline) - mu_total = residual - mu_total
        delta_target = hr_res - mu_total
        valid_mask = torch.isfinite(hr_res)

        # Strip batch dim (1) since dataset items are per-sample.
        mu_HR_list.append(mu_total.squeeze(0).cpu())
        baseline_log_list.append(bl.squeeze(0).cpu())
        delta_target_list.append(torch.nan_to_num(delta_target, nan=0.0).squeeze(0).cpu())
        valid_mask_list.append(valid_mask.squeeze(0).cpu())

        if (i + 1) % 500 == 0:
            _elapsed = time.time() - _t0
            _rate = (i + 1) / max(_elapsed, 1e-6)
            print(f'  {i + 1}{f"/{_n_iter}" if _n_iter else ""} cached '
                  f'| {_elapsed / 60:.1f} min | {_rate:.1f} samp/s')

    cache = {
        'mu_HR':        torch.stack(mu_HR_list, dim=0),
        'baseline_log': torch.stack(baseline_log_list, dim=0),
        'delta_target': torch.stack(delta_target_list, dim=0),
        'valid_mask':   torch.stack(valid_mask_list, dim=0),
    }
    STAGE1_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(cache, STAGE1_CACHE_PATH)
    print(f'[Cell 5] Cache saved : {STAGE1_CACHE_PATH}')

print(f'[Cell 5] Cache shapes :')
print(f'  mu_HR        = {tuple(cache["mu_HR"].shape)}')
print(f'  baseline_log = {tuple(cache["baseline_log"].shape)}')
print(f'  delta_target = {tuple(cache["delta_target"].shape)}')
print(f'  valid_mask   = {tuple(cache["valid_mask"].shape)}')
print(f'[Cell 5] delta_target stats : mean={cache["delta_target"].mean():.4f} '
      f'std={cache["delta_target"].std():.4f}')


# Map-style dataset (yields dicts compatible with train_epoch_stage2_cached).
class _CachedDataset(torch.utils.data.Dataset):
    def __init__(self, cache, indices=None):
        self.mu_HR        = cache['mu_HR']
        self.baseline_log = cache['baseline_log']
        self.delta_target = cache['delta_target']
        self.valid_mask   = cache['valid_mask']
        self.indices = indices if indices is not None else list(range(len(self.mu_HR)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            'mu_HR':        self.mu_HR[idx],
            'baseline_log': self.baseline_log[idx],
            'delta_target': self.delta_target[idx],
            'valid_mask':   self.valid_mask[idx],
        }


N_total = int(cache['mu_HR'].shape[0])
VAL_FRACTION = 0.1
N_train = int(N_total * (1.0 - VAL_FRACTION))
train_indices = list(range(N_train))
val_indices   = list(range(N_train, N_total))

train_cached_dataset = _CachedDataset(cache, train_indices)
val_cached_dataset   = _CachedDataset(cache, val_indices)

STAGE2_BATCH_SIZE = 64
train_cached_dataloader = torch.utils.data.DataLoader(
    train_cached_dataset,
    batch_size=STAGE2_BATCH_SIZE,
    shuffle=True,
    num_workers=0,         # was 4 - workers cause stall on mmap'd cache
    pin_memory=False,      # was True - pin on mmap triggers memcpy each access
    drop_last=False,
)
val_cached_dataloader = torch.utils.data.DataLoader(
    val_cached_dataset,
    batch_size=STAGE2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)
print(f'[Cell 5] Splits  : train={len(train_cached_dataset)}  val={len(val_cached_dataset)}')
print(f'[Cell 5] Loaders : train_bs={STAGE2_BATCH_SIZE}  val_bs={STAGE2_BATCH_SIZE}')


In [ ]:
# === Cell 6 : Build Stage 2 (warm-start) + EMA ===
import copy
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

# Probe HR channels (1 for precip).
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])
print(f'[Cell 6] hr_channels = {hr_channels}')


def build_stage2(warm_start: bool, gradient_checkpointing: bool):
    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
    UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ('down_block_types', 'up_block_types'):
        if _k in UNET_KW and isinstance(UNET_KW[_k], list):
            UNET_KW[_k] = tuple(UNET_KW[_k])
    UNET_KW['projection_class_embeddings_input_dim'] = (
        num_vars * int(CONFIG.diffusion.conditioning_dim))

    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET_KW,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(gradient_checkpointing),
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)

    if warm_start:
        if not CKPT_STAGE2_WARMSTART.exists():
            raise FileNotFoundError(f'Warm-start ckpt missing : {CKPT_STAGE2_WARMSTART}')
        ck = torch.load(CKPT_STAGE2_WARMSTART, map_location=DEVICE, weights_only=False)
        diff_sd = _strip_prefixes(ck.get('diffusion_state_dict'))
        if diff_sd is None:
            raise RuntimeError(f'diffusion_state_dict absent from {CKPT_STAGE2_WARMSTART}')
        info = diff.load_state_dict(diff_sd, strict=False)
        print(f'  [build_stage2] warm-start loaded | missing={len(info.missing_keys)} '
              f'unexpected={len(info.unexpected_keys)}')
        del ck, diff_sd
    else:
        print('  [build_stage2] from scratch (random init)')

    diff.edm_config.sigma_data = float(SIGMA_DATA_NEW)
    print(f'  [build_stage2] sigma_data set to {diff.edm_config.sigma_data}')

    for p in diff.parameters():
        p.requires_grad_(True)
    diff.train()
    _n = sum(p.numel() for p in diff.parameters())
    print(f'  [build_stage2] diffusion params : {_n:,}')
    return diff


# Build live + EMA decoders.
diffusion_decoder = build_stage2(WARM_START, GRADIENT_CHECKPOINTING)

diffusion_ema = build_stage2(WARM_START, gradient_checkpointing=False)
# EMA mirrors live at init.
diffusion_ema.load_state_dict(diffusion_decoder.state_dict())
diffusion_ema.eval()
for _p in diffusion_ema.parameters():
    _p.requires_grad_(False)
print(f'[Cell 6] live + EMA decoders ready (EMA decay = {EMA_DECAY})')


In [ ]:
# === Cell 7 : Training loop -- 50 ep fine-tune with EMA + persist + resume ===
import time, gc
from st_cdgm.training.two_stage import train_epoch_stage2_cached

# Optimizer + scheduler.
optimizer = torch.optim.AdamW(
    diffusion_decoder.parameters(),
    lr=LR, betas=(BETA1, BETA2), weight_decay=WEIGHT_DECAY,
)
scheduler = None
if WARMUP_STEPS > 0:
    from torch.optim.lr_scheduler import LinearLR
    scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                         total_iters=int(WARMUP_STEPS))

# Resume state.
start_epoch = 1
best_loss = float('inf')
training_history = []

if RESUME and CKPT_LAST.exists():
    print(f'[Cell 7] RESUME from {CKPT_LAST}')
    _ck = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    diffusion_decoder.load_state_dict(_strip_prefixes(_ck['diffusion_state_dict']))
    if _ck.get('diffusion_ema_state_dict') is not None:
        diffusion_ema.load_state_dict(_strip_prefixes(_ck['diffusion_ema_state_dict']))
    if _ck.get('optimizer_state_dict') is not None:
        try:
            optimizer.load_state_dict(_ck['optimizer_state_dict'])
        except Exception as _e:
            print(f'  [warn] optimizer resume failed : {_e}')
    start_epoch = int(_ck.get('epoch', 0)) + 1
    best_loss = float(_ck.get('best_loss', float('inf')))
    training_history = list(_ck.get('training_history', []))
    print(f'  Resumed at epoch {start_epoch}/{EPOCHS_TARGET} | best_loss={best_loss:.5f} '
          f'| history={len(training_history)} entries')
    del _ck
else:
    print(f'[Cell 7] No resume : starting from epoch 1 / {EPOCHS_TARGET}')


def _ema_update(ema_model, live_model, decay):
    """Karras EDM2 EMA : parameters mul+add, buffers copy."""
    with torch.no_grad():
        for p_ema, p_live in zip(ema_model.parameters(), live_model.parameters()):
            p_ema.data.mul_(decay).add_(p_live.data, alpha=1.0 - decay)
        for b_ema, b_live in zip(ema_model.buffers(), live_model.buffers()):
            b_ema.data.copy_(b_live.data)


print(f'\n[Cell 7] Training from epoch {start_epoch} to {EPOCHS_TARGET}')

for ep in range(start_epoch, EPOCHS_TARGET + 1):
    _t0 = time.time()

    # train_epoch_stage2_cached drives the inner loop (bf16 auto on A100).
    # We pass ema_model so the EMA update happens inside the loop after each step.
    metrics = train_epoch_stage2_cached(
        diffusion_decoder=diffusion_decoder,
        optimizer=optimizer,
        cached_dataloader=train_cached_dataloader,
        device=DEVICE,
        use_amp=USE_AMP,
        gradient_clipping=GRADIENT_CLIP,
        log_every=20,
        verbose=True,
        lambda_contrastive_dag=0.0,
        ema_model=diffusion_ema,         # in-loop EMA update
        ema_decay=EMA_DECAY,
        ema_warmup_steps=0,
        conditioning_dropout_prob=0.0,
        log_loss_components=False,        # smoke proved this is unnecessary noise
    )

    if scheduler is not None:
        scheduler.step()

    ep_time = time.time() - _t0
    train_loss = float(metrics.get('loss_diff', float('nan')))
    print(f'  [ep{ep}/{EPOCHS_TARGET}] train_loss={train_loss:.5f}  '
          f'time={ep_time:.1f}s')

    _record = {
        'epoch': ep, 'train_loss': train_loss, 'epoch_time_s': ep_time,
    }
    training_history.append(_record)

    # Persist LAST checkpoint every epoch.
    is_best = train_loss < best_loss
    if is_best:
        best_loss = train_loss

    payload = {
        'epoch': ep,
        'diffusion_state_dict':     diffusion_decoder.state_dict(),
        'diffusion_ema_state_dict': diffusion_ema.state_dict(),
        'optimizer_state_dict':     optimizer.state_dict(),
        'best_loss':                best_loss,
        'training_history':         training_history,
        'sigma_data':               float(SIGMA_DATA_NEW),
        'warm_start':               bool(WARM_START),
        'epochs_target':            int(EPOCHS_TARGET),
        'lr':                       float(LR),
        'ema_decay':                float(EMA_DECAY),
    }
    torch.save(payload, CKPT_LAST)

    if is_best:
        if CKPT_BEST.exists():
            import shutil as _shutil
            _shutil.copy(str(CKPT_BEST), str(CKPT_BEST) + '.bak')
        torch.save(payload, CKPT_BEST)
        print(f'  [ep{ep}] * NEW BEST train_loss = {best_loss:.5f} -> {CKPT_BEST.name}')

    HISTORY_JSON.write_text(
        json.dumps(training_history, indent=2, default=str), encoding='utf-8'
    )

    # Light gc between epochs (no del/empty_cache : we keep state across epochs).
    gc.collect()

print(f'\n[Cell 7] Training done. epochs ran {start_epoch}..{EPOCHS_TARGET}.'
      f'  best_loss={best_loss:.5f}')
print(f'[Cell 7] Final checkpoint : {CKPT_LAST}')
print(f'[Cell 7] Best  checkpoint : {CKPT_BEST}')
print(f'[Cell 7] History JSON     : {HISTORY_JSON}')


In [ ]:
# === Cell 8 : NEXT STEP -- BS30 final eval (separate notebook) ===
#
# To evaluate this fine-tuned Stage 2 with the full BS30 protocol :
#   1. Open phase6_dualpath_final_validation.ipynb
#   2. In its Cell 2 (Constants), point STAGE2_CHECKPOINT to :
print()
print('=' * 78)
print('NEXT STEP : BS30 eval via phase6_dualpath_final_validation.ipynb')
print('=' * 78)
print()
print(f'STAGE2_CHECKPOINT = "{CKPT_BEST}"  # use EMA via the load helper')
print()
print('Or use epoch_last.pth if you want the latest (non-best) state :')
print(f'STAGE2_CHECKPOINT = "{CKPT_LAST}"')
print()
print('That notebook produces :')
print('  - final_validation_metrics.json   (Pearson, RMSE, F1@p99, CRPS)')
print('  - domain_metrics.json             (CDD, Rx1Day, R10mm, spatial RAPSD)')
print('  - 3-way comparison vs causal V5 + non-causal CorrDiff')
